In [ ]:
"""
01_salary_prediction.ipynb
Jupyter Notebook: Linear Regression for Salary Prediction

This notebook demonstrates:
1. Data loading and exploration
2. Feature engineering and preprocessing
3. Linear regression from scratch
4. Comparison with scikit-learn
5. Model evaluation and visualization
"""

# Cell 1: Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression as SklearnLR
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import sys
sys.path.append('../src')
from models import LinearRegressionScratch, ModelVisualizer, calculate_metrics
from preprocessing import DataPreprocessor, generate_salary_dataset

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("Libraries imported successfully!")

# Cell 2: Generate and Load Data
print("="*70)
print("SALARY PREDICTION PROJECT")
print("="*70)

# Generate synthetic salary data
df = generate_salary_dataset(n_samples=500, save_path='../data/raw/salary_data.csv')

print("\nDataset Info:")
print(f"Shape: {df.shape}")
print(f"\nFirst 5 rows:")
print(df.head())

print(f"\nDataset Statistics:")
print(df.describe())

print(f"\nMissing Values:")
print(df.isnull().sum())

# Cell 3: Exploratory Data Analysis
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Salary Dataset - Exploratory Data Analysis', fontsize=16, fontweight='bold')

# Distribution of target
axes[0, 0].hist(df['Salary'], bins=30, edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Salary ($)', fontsize=11)
axes[0, 0].set_ylabel('Frequency', fontsize=11)
axes[0, 0].set_title('Salary Distribution', fontweight='bold')
axes[0, 0].axvline(df['Salary'].mean(), color='red', linestyle='--', 
                   label=f'Mean: ${df["Salary"].mean():,.0f}')
axes[0, 0].legend()

# Years Experience vs Salary
axes[0, 1].scatter(df['YearsExperience'], df['Salary'], alpha=0.6, s=50, edgecolors='k')
axes[0, 1].set_xlabel('Years of Experience', fontsize=11)
axes[0, 1].set_ylabel('Salary ($)', fontsize=11)
axes[0, 1].set_title('Experience vs Salary', fontweight='bold')

# Education Level vs Salary
axes[0, 2].boxplot([df[df['EducationLevel']==i]['Salary'] for i in range(1, 5)],
                   labels=['High School', 'Bachelor', 'Master', 'PhD'])
axes[0, 2].set_xlabel('Education Level', fontsize=11)
axes[0, 2].set_ylabel('Salary ($)', fontsize=11)
axes[0, 2].set_title('Education vs Salary', fontweight='bold')
axes[0, 2].tick_params(axis='x', rotation=45)

# Age vs Salary
axes[1, 0].scatter(df['Age'], df['Salary'], alpha=0.6, s=50, edgecolors='k', c='green')
axes[1, 0].set_xlabel('Age', fontsize=11)
axes[1, 0].set_ylabel('Salary ($)', fontsize=11)
axes[1, 0].set_title('Age vs Salary', fontweight='bold')

# Hours per Week vs Salary
axes[1, 1].scatter(df['HoursPerWeek'], df['Salary'], alpha=0.6, s=50, edgecolors='k', c='purple')
axes[1, 1].set_xlabel('Hours per Week', fontsize=11)
axes[1, 1].set_ylabel('Salary ($)', fontsize=11)
axes[1, 1].set_title('Work Hours vs Salary', fontweight='bold')

# Correlation heatmap
axes[1, 2].clear()
corr_matrix = df.corr()
im = axes[1, 2].imshow(corr_matrix, cmap='coolwarm', aspect='auto', vmin=-1, vmax=1)
axes[1, 2].set_xticks(range(len(corr_matrix.columns)))
axes[1, 2].set_yticks(range(len(corr_matrix.columns)))
axes[1, 2].set_xticklabels(corr_matrix.columns, rotation=45, ha='right')
axes[1, 2].set_yticklabels(corr_matrix.columns)
axes[1, 2].set_title('Correlation Matrix', fontweight='bold')

# Add correlation values
for i in range(len(corr_matrix)):
    for j in range(len(corr_matrix)):
        text = axes[1, 2].text(j, i, f'{corr_matrix.iloc[i, j]:.2f}',
                              ha="center", va="center", color="black", fontsize=9)

plt.colorbar(im, ax=axes[1, 2])
plt.tight_layout()
plt.show()

print("\nCorrelation with Salary:")
print(corr_matrix['Salary'].sort_values(ascending=False))

# Cell 4: Data Preprocessing
print("\n" + "="*70)
print("DATA PREPROCESSING")
print("="*70)

# Initialize preprocessor
preprocessor = DataPreprocessor(scaling_method='standard')

# Prepare features and target
X, y = preprocessor.prepare_features(df, target_column='Salary')

print(f"\nFeature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")
print(f"\nFeatures: {preprocessor.feature_names}")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\nTraining set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

# Cell 5: Train Linear Regression from Scratch
print("\n" + "="*70)
print("TRAINING LINEAR REGRESSION FROM SCRATCH")
print("="*70)

# Initialize and train model
model_scratch = LinearRegressionScratch(
    learning_rate=0.01,
    n_iterations=2000,
    tolerance=1e-6
)

print("\nTraining model...")
model_scratch.fit(X_train, y_train, verbose=True)

# Make predictions
y_pred_scratch = model_scratch.predict(X_test)

# Calculate metrics
print("\n" + "="*70)
print("MODEL FROM SCRATCH - PERFORMANCE METRICS")
print("="*70)
metrics_scratch = calculate_metrics(y_test, y_pred_scratch)
for metric, value in metrics_scratch.items():
    print(f"{metric:10s}: {value:,.4f}")

# Display model parameters
print(f"\nModel Parameters:")
print(f"Weights: {model_scratch.weights}")
print(f"Bias: {model_scratch.bias:.4f}")

# Cell 6: Visualize Convergence
print("\n" + "="*70)
print("VISUALIZING GRADIENT DESCENT CONVERGENCE")
print("="*70)

visualizer = ModelVisualizer()
fig = visualizer.plot_cost_history(model_scratch, figsize=(14, 5))
plt.show()

print(f"\nInitial Cost: {model_scratch.cost_history[0]:.4f}")
print(f"Final Cost: {model_scratch.cost_history[-1]:.4f}")
print(f"Cost Reduction: {model_scratch.cost_history[0] - model_scratch.cost_history[-1]:.4f}")
print(f"Iterations to Converge: {len(model_scratch.cost_history)}")

# Cell 7: Train Scikit-Learn Model
print("\n" + "="*70)
print("TRAINING SCIKIT-LEARN LINEAR REGRESSION")
print("="*70)

# Initialize and train scikit-learn model
model_sklearn = SklearnLR()
model_sklearn.fit(X_train, y_train)

# Make predictions
y_pred_sklearn = model_sklearn.predict(X_test)

# Calculate metrics
print("\nSCIKIT-LEARN MODEL - PERFORMANCE METRICS")
print("="*70)
metrics_sklearn = calculate_metrics(y_test, y_pred_sklearn)
for metric, value in metrics_sklearn.items():
    print(f"{metric:10s}: {value:,.4f}")

print(f"\nModel Parameters:")
print(f"Weights: {model_sklearn.coef_}")
print(f"Bias: {model_sklearn.intercept_:.4f}")

# Cell 8: Compare Models
print("\n" + "="*70)
print("MODEL COMPARISON")
print("="*70)

# Create comparison dataframe
comparison_df = pd.DataFrame({
    'Metric': ['MSE', 'RMSE', 'MAE', 'R²'],
    'From Scratch': [metrics_scratch['MSE'], metrics_scratch['RMSE'], 
                     metrics_scratch['MAE'], metrics_scratch['R²']],
    'Scikit-Learn': [metrics_sklearn['MSE'], metrics_sklearn['RMSE'],
                     metrics_sklearn['MAE'], metrics_sklearn['R²']]
})

print(comparison_df.to_string(index=False))

# Parameter comparison
print("\n" + "="*70)
print("PARAMETER COMPARISON")
print("="*70)
print(f"\n{'Feature':<20} {'From Scratch':>15} {'Scikit-Learn':>15} {'Difference':>15}")
print("-" * 70)
for i, feature in enumerate(preprocessor.feature_names):
    diff = abs(model_scratch.weights[i] - model_sklearn.coef_[i])
    print(f"{feature:<20} {model_scratch.weights[i]:>15.6f} {model_sklearn.coef_[i]:>15.6f} {diff:>15.6f}")

bias_diff = abs(model_scratch.bias - model_sklearn.intercept_)
print(f"{'Bias':<20} {model_scratch.bias:>15.6f} {model_sklearn.intercept_:>15.6f} {bias_diff:>15.6f}")

# Cell 9: Visualize Predictions
print("\n" + "="*70)
print("PREDICTION VISUALIZATIONS")
print("="*70)

# Predictions vs Actual
fig = visualizer.compare_models(
    {'From Scratch': model_scratch, 'Scikit-Learn': model_sklearn},
    X_test, y_test,
    figsize=(14, 6)
)
plt.show()

# Cell 10: Residual Analysis
print("\n" + "="*70)
print("RESIDUAL ANALYSIS")
print("="*70)

# From Scratch residuals
fig_scratch = visualizer.plot_residuals(y_test, y_pred_scratch, figsize=(14, 5))
fig_scratch.suptitle('From Scratch Model - Residual Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.show()

# Scikit-Learn residuals
fig_sklearn = visualizer.plot_residuals(y_test, y_pred_sklearn, figsize=(14, 5))
fig_sklearn.suptitle('Scikit-Learn Model - Residual Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.show()

# Cell 11: Feature Importance
print("\n" + "="*70)
print("FEATURE IMPORTANCE")
print("="*70)

# Create feature importance plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# From Scratch
importance_scratch = np.abs(model_scratch.weights)
sorted_idx_scratch = np.argsort(importance_scratch)[::-1]
ax1.barh(range(len(importance_scratch)), importance_scratch[sorted_idx_scratch], color='skyblue', edgecolor='black')
ax1.set_yticks(range(len(importance_scratch)))
ax1.set_yticklabels([preprocessor.feature_names[i] for i in sorted_idx_scratch])
ax1.set_xlabel('Absolute Weight Value', fontsize=12)
ax1.set_title('From Scratch - Feature Importance', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='x')

# Scikit-Learn
importance_sklearn = np.abs(model_sklearn.coef_)
sorted_idx_sklearn = np.argsort(importance_sklearn)[::-1]
ax2.barh(range(len(importance_sklearn)), importance_sklearn[sorted_idx_sklearn], color='lightcoral', edgecolor='black')
ax2.set_yticks(range(len(importance_sklearn)))
ax2.set_yticklabels([preprocessor.feature_names[i] for i in sorted_idx_sklearn])
ax2.set_xlabel('Absolute Coefficient Value', fontsize=12)
ax2.set_title('Scikit-Learn - Feature Importance', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

# Print importance ranking
print("\nFeature Importance Ranking (From Scratch):")
for i, idx in enumerate(sorted_idx_scratch, 1):
    print(f"{i}. {preprocessor.feature_names[idx]}: {importance_scratch[idx]:.6f}")

# Cell 12: Make Predictions on New Data
print("\n" + "="*70)
print("PREDICTIONS ON NEW DATA")
print("="*70)

# Create sample new data
new_data = pd.DataFrame({
    'YearsExperience': [2, 5, 10],
    'EducationLevel': [2, 3, 4],
    'Age': [25, 30, 35],
    'HoursPerWeek': [40, 45, 50]
})

print("\nNew Data:")
print(new_data)

# Prepare new data
X_new, _ = preprocessor.prepare_features(
    pd.concat([df, new_data]).tail(3),
    target_column='Salary',
    fit=False
)

# Make predictions
predictions_scratch = model_scratch.predict(X_new)
predictions_sklearn = model_sklearn.predict(X_new)

print("\n" + "="*70)
print("PREDICTIONS")
print("="*70)
print(f"\n{'Experience':<12} {'Education':<10} {'Age':<5} {'Hours':<7} {'Predicted (Scratch)':<20} {'Predicted (Sklearn)':<20}")
print("-" * 90)
for i in range(len(new_data)):
    print(f"{new_data.iloc[i]['YearsExperience']:<12.1f} "
          f"{new_data.iloc[i]['EducationLevel']:<10.0f} "
          f"{new_data.iloc[i]['Age']:<5.1f} "
          f"{new_data.iloc[i]['HoursPerWeek']:<7.1f} "
          f"${predictions_scratch[i]:<19,.2f} "
          f"${predictions_sklearn[i]:<19,.2f}")

# Cell 13: Summary and Conclusions
print("\n" + "="*70)
print("PROJECT SUMMARY")
print("="*70)

print("""
This project demonstrated:

1. ✓ Linear Regression implementation from scratch using gradient descent
2. ✓ Data preprocessing and feature engineering
3. ✓ Model training and convergence visualization
4. ✓ Comparison with scikit-learn implementation
5. ✓ Comprehensive model evaluation
6. ✓ Residual analysis and diagnostics

Key Findings:
- Both models achieve similar performance (R² ≈ {:.4f})
- Gradient descent successfully converged in {} iterations
- Model parameters match closely between implementations
- Most important features: {}

Next Steps:
- Try polynomial features for non-linear relationships
- Experiment with regularization (Ridge/Lasso)
- Cross-validation for robust performance estimation
- Feature selection techniques
""".format(
    metrics_scratch['R²'],
    len(model_scratch.cost_history),
    ', '.join([preprocessor.feature_names[i] for i in sorted_idx_scratch[:3]])
))

print("="*70)
print("PROJECT COMPLETED SUCCESSFULLY!")
print("="*70)